# Costruiamo una pipeline ETL: da CSV a database SQLite

Portiamo tre serie storiche di mercato (indice azionario mondiale, indice azionario
mondiale small cap, oro) da file CSV a un database SQLite, con una pipeline in tre
fasi: **Extract → Transform → Load**. Poi interroghiamo il database con SQL, e
infine (bonus) raffiniamo i dati tagliando il periodo iniziale in cui non tutte le
fonti erano ancora disponibili.


## Setup

Importiamo le librerie che useremo.

In [ ]:
import pandas as pd
import sqlite3


## Configurazione

`elenco` associa a ciascuna fonte (il nome che vogliamo dare alla colonna) il
percorso del file CSV corrispondente. Tutti i file sono ospitati in un repository
GitHub, così tutti lavoriamo esattamente sugli stessi dati.


In [ ]:
url = "https://raw.githubusercontent.com/GabrieleL98/ADABI_Python_LAB/main/"

elenco = {
    "WORLD": "world.csv",
    "WORLD_SMALL_CAP": "world_small_cap.csv",
    "GOLD": "gold.csv",
}

#il nome del nostro futuro db
db_path = "mercati.db"


## Extract

Ogni file CSV ha due colonne: la prima è la data, la seconda il valore. Vogliamo
unire tutte le fonti in un solo DataFrame "wide" (una riga per data, una colonna
per fonte), allineate automaticamente per data.

Un modo naturale: per ciascuna fonte leggiamo il CSV, lo trasformiamo in una
`Series` indicizzata per data, e raccogliamo tutte le Series in un dizionario.
Passare un dizionario `{nome_colonna: serie}` a `pd.DataFrame(...)` costruisce
esattamente la tabella che vogliamo, allineando automaticamente le date.


In [ ]:
def extract(elenco, url):
    """
    Legge ogni file CSV elencato in `elenco` (nome_colonna -> percorso_file) e le
    unisce in un solo DataFrame "wide": una riga per data, una colonna per fonte.
    """
    serie = {}
    for name, path in elenco.items():
        df_temp = pd.read_csv(
            url + path,
            usecols=[0, 1],
            names=['Date', name],
            header=0,
            parse_dates=['Date'],
        )
        # trasformiamo il DataFrame a due colonne in una Series indicizzata per data
        serie[name] = df_temp.set_index('Date')[name]

    # pd.DataFrame su un dizionario di Series allinea automaticamente per indice
    # (la data). Dove una fonte non ha quella data, per ora resta NaN: ce ne
    # occupiamo nel prossimo passo (Transform).
    df_combined = pd.DataFrame(serie).sort_index()
    return df_combined


**Un modo alternativo, altrettanto valido**: invece del dizionario di Series,
si può partire dal primo DataFrame letto e unire gli altri uno alla volta con
`pd.merge(..., on='Date', how='outer')`. È più vicino al concetto di *JOIN* che
incontrerete parlando di database: "unisci due tabelle sulla colonna Date,
tenendo tutte le date di entrambe" (`how='outer'`).


In [ ]:
def extract_merge(elenco, url):
    """Stessa idea di extract(), ma unendo le fonti una alla volta con un merge
    esplicito su Date invece di passare per un dizionario di Series."""
    df_combined = None
    for name, path in elenco.items():
        df_temp = pd.read_csv(
            url + path, usecols=[0, 1], names=['Date', name], header=0, parse_dates=['Date'],
        )
        if df_combined is None:
            df_combined = df_temp
        else:
            df_combined = pd.merge(df_combined, df_temp, on='Date', how='outer')
    return df_combined.set_index('Date').sort_index()


In [ ]:
# Verifica risultati
df_raw = extract(elenco, url)
df_raw.head()

,WORLD,WORLD_SMALL_CAP,GOLD
Date,,,
1998-10-06,NaN,NaN,245.20
1998-10-07,NaN,NaN,247.74
1998-10-08,NaN,NaN,244.81
1998-10-09,NaN,NaN,245.05
1998-10-12,NaN,NaN,247.39


## Transform

Le fonti non hanno tutte le stesse date (festivi diversi, giorni mancanti).
Costruiamo un calendario giornaliero continuo (tutti i giorni, weekend compresi)
e riportiamo avanti (*forward-fill*) l'ultimo valore conosciuto per riempire i
buchi — es. un sabato prende il valore del venerdì precedente.

**Attenzione ai NaN iniziali**: GOLD ha dati dal 1998, ma WORLD e WORLD_SMALL_CAP
solo dal 2000. Per quelle prime righe, WORLD e WORLD_SMALL_CAP resteranno `NaN`
anche dopo il forward-fill (non c'è "un valore precedente" da riportare avanti,
perché quell'indice semplicemente non esisteva ancora). **Questo è corretto e
voluto** per ora: non inventiamo un valore per un indice che non esisteva, e non
buttiamo via la storia valida di GOLD solo perché un'altra colonna è vuota. Più
avanti, come esercizio bonus, vedremo come e quando decidere di tagliare questo
periodo iniziale.


In [ ]:
def transform(df_combined):
    """
    Crea un calendario giornaliero continuo tra la prima e l'ultima data
    disponibile, e riporta avanti (ffill) l'ultimo valore noto per riempire i
    giorni mancanti.
    """
    all_days = pd.date_range(
        start=df_combined.index.min(),
        end=df_combined.index.max(),
        freq='D',
    )
    df_final = df_combined.reindex(all_days).ffill()
    df_final.index.name = 'Data'
    return df_final

In [ ]:
# mostriamo i risultati
df_clean = transform(df_raw)
df_clean.head()

,WORLD,WORLD_SMALL_CAP,GOLD
Data,,,
1998-10-06,NaN,NaN,245.20
1998-10-07,NaN,NaN,247.74
1998-10-08,NaN,NaN,244.81
1998-10-09,NaN,NaN,245.05
1998-10-10,NaN,NaN,245.05


## Load

Scriviamo `df_clean` in una tabella SQLite chiamata `prices`. Punto chiave: se
rilanciamo lo script più volte NON vogliamo che le righe si accumulino
all'infinito. Per questo:

- la colonna `Date` è **PRIMARY KEY**: SQLite non può avere due righe con la
  stessa data;
- scriviamo con **`INSERT OR REPLACE`**: se la data esiste già, la riga viene
  sovrascritta con i nuovi valori; altrimenti viene creata.

Risultato: rilanciare lo script quante volte vogliamo produce sempre lo stesso
identico contenuto nel database — nessun duplicato, nessuna crescita infinita.


In [ ]:
def load(df_final, db_path):
    conn = sqlite3.connect(db_path)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS prices (
            Date TEXT PRIMARY KEY,
            WORLD REAL,
            WORLD_SMALL_CAP REAL,
            GOLD REAL
        )
    """)

    # rimette il classico indice 0, 1, 2... trasformando l'indice "data" a colonna
    df_da_scrivere = df_clean.reset_index()

    # trasforma il formato data in formato standard ISO SQLite AAAA-MM-DD
    df_da_scrivere['Data'] = df_da_scrivere['Data'].dt.strftime('%Y-%m-%d')

    # operazione di insert righe
    conn.executemany(
        """
        INSERT OR REPLACE INTO prices (Date, WORLD, WORLD_SMALL_CAP, GOLD)
        VALUES (?, ?, ?, ?)
        """,
        # (?, ? , ?, ?) è un placeholder necesasrio per il funzionamento della query
        # trasforma le colonne in una matrice numpy con "[[]]"" e poi in liste, step necessario per passarle come VALUES a SQL
        df_da_scrivere[['Data', 'WORLD', 'WORLD_SMALL_CAP', 'GOLD']].values.tolist(),
    )

    conn.commit()
    conn.close()

load(df_clean, db_path)
print(f"Caricate (prima funzione) {len(df_clean)} righe nel database {db_path}.")


# Oppure, più breve ma meno "parlante":
def load(df_final, db_path):
    conn = sqlite3.connect(db_path)
    df_final.to_sql("prices", conn, if_exists="replace", index=True, index_label="Date")
    conn.close()

load(df_clean, db_path)
print(f"Caricate (seconda funzione) {len(df_clean)} righe nel database {db_path}.")


Caricate (prima funzione) 10161 righe nel database mercati.db.
Caricate (seconda funzione) 10161 righe nel database mercati.db.


**Verifica di idempotenza**: eseguite di nuovo la cella sopra (Load). Poi
controllate quante righe ci sono nella tabella — deve essere rimasto lo stesso
numero.


In [ ]:
conn = sqlite3.connect(db_path)
totale = conn.execute("SELECT COUNT(*) FROM prices").fetchone()[0]
print("Righe totali nella tabella:", totale)
conn.close()


Righe totali nella tabella: 10161


---
## Mettiamo tutto insieme

Finora abbiamo eseguito `extract`, `transform` e `load` in celle separate.
Scriviamo ora un'unica funzione che le richiama in sequenza: partendo solo
dalla configurazione (`elenco`, `url`, `db_path`), costruisce da sola l'intero
database. Comoda per rilanciare tutta la pipeline in un colpo solo — è anche
esattamente il codice che mettereste in un file `.py` a sé stante se voleste
eseguire l'ETL da riga di comando invece che da notebook.


In [ ]:
def run_etl(elenco, url, db_path):
    """Esegue l'intera pipeline ETL: extract -> transform -> load."""
    df_raw = extract(elenco, url)
    df_clean = transform(df_raw)
    load(df_clean, db_path)
    print(f"Pipeline completata: {len(df_clean)} righe scritte in {db_path}.")


In [ ]:
run_etl(elenco, url, db_path)


Pipeline completata: 10161 righe scritte in mercati.db.


---
## Interroghiamo il database con SQL

Ora che i dati sono in un database, possiamo interrogarlo con query SQL invece
di ricaricare i CSV. Ci sono due modi comuni per farlo da Python:

1. **`cursor.execute(...).fetchall()`**: esegui la query, ottieni i risultati
   come lista di tuple.
2. **`pd.read_sql(...)`**: esegui la query e ottieni direttamente un DataFrame
   pandas — più comodo quando i risultati sono tabellari.

Proviamo entrambi.


In [ ]:
conn = sqlite3.connect(db_path)

# Modo 1: cursor.execute + fetchall
cur = conn.cursor()
cur.execute("SELECT MIN(Date), MAX(Date) FROM prices")
print("Intervallo di date nel database:", cur.fetchone())

cur.execute("SELECT * FROM prices ORDER BY Date DESC LIMIT 5")
print("\nUltime 5 righe:")
for riga in cur.fetchall():
    print(riga)

conn.close()


Intervallo di date nel database: ('1998-10-06 00:00:00', '2026-07-31 00:00:00')

Ultime 5 righe:
('2026-07-31 00:00:00', 697.0335567851051, 787.9290126011891, 4006.6)
('2026-07-30 00:00:00', 693.3740335218727, 789.7626032472773, 4006.6)
('2026-07-29 00:00:00', 689.5185645243995, 785.7405730611422, 4006.6)
('2026-07-28 00:00:00', 697.3408747306012, 794.3295290057933, 4006.6)
('2026-07-27 00:00:00', 697.6115570015975, 798.2021356936978, 4006.6)


In [ ]:
conn = sqlite3.connect(db_path)

# Modo 2: pd.read_sql -> risultato direttamente come DataFrame
media_gold_2020 = pd.read_sql(
    "SELECT AVG(GOLD) AS media_gold FROM prices WHERE Date >= '2020-01-01'",
    conn,
)
print("Prezzo medio di GOLD dal 2020 in poi:")
display(media_gold_2020)

conn.close()


Prezzo medio di GOLD dal 2020 in poi:


,media_gold
0,2156.604097


### La stessa domanda, due strade

Molte domande si possono risolvere sia con una query SQL sul database, sia
direttamente con pandas sul DataFrame che avete già in memoria (`df_clean`).
Sotto il cofano fanno un lavoro molto simile, ciò che cambia è solo il linguaggio in cui lo scrivete.

Due esempi:

In [ ]:
conn = sqlite3.connect(db_path)

# Domanda 1: qual è il valore massimo di GOLD in tutto il periodo?
max_gold_sql = pd.read_sql("SELECT MAX(GOLD) AS max_gold FROM prices", conn).iloc[0, 0] #iloc[0,0] serve solo per estrarre il contenuto all'interno della cella, logicamente il codice funzionerebbe anche senza
max_gold_pandas = df_clean["GOLD"].max()
print("Massimo GOLD - SQL:", max_gold_sql, " | pandas:", max_gold_pandas)

# Domanda 2: in quanti giorni WORLD ha superato 600?
giorni_sopra_600_sql = pd.read_sql(
    "SELECT COUNT(*) AS n FROM prices WHERE WORLD > 600", conn
).iloc[0, 0]
giorni_sopra_600_pandas = (df_clean["WORLD"] > 600).sum()
print("Giorni con WORLD > 600 - SQL:", giorni_sopra_600_sql, " | pandas:", giorni_sopra_600_pandas)

conn.close()


Massimo GOLD - SQL: 4552.98  | pandas: 4552.98
Giorni con WORLD > 600 - SQL: 314  | pandas: 314


# Esercizi

Rispondi alle seguenti domande:

1.   In quanti giorni WORLD ha superato i 600 punti?
2.   Qual è il valore massimo di WORLD nel 2000?
3.   Qual è il minimo valore raggiunto da GOLD tra il 04/08/2016 e il 31/12/2017?
4.   Qual è il valore medio di ciascun indice per ciascun anno?
5.   In quale anno la volatilità (deviazione standard) di WORLD_SMALL_CAP è stata più alta?
6.   Qual è la correlazione tra WORLD e GOLD?
7.   In quanti giorni WORLD ha chiuso più in alto rispetto al giorno precedente?

Nota che ora che abbiamo un DB e un modo per interrogarlo tramite python potete rispondere sia tramite python che tramite SQL. Trovo sia interessante provare ad utilizzare entrambi i metodi. Da un lato un linguaggio più semplice e "naturale" dall'altro uno più complesso ma molto potente.

Ad alcune di queste domande sarà più facile rispondere usando SQL ma man mano che si avanza con la complessità della richiesta python vince a mani basse.

**domanda numero 1**

In [ ]:
(df_clean['WORLD'] > 600).sum()

'SELECT COUNT(*) FROM prices WHERE WORLD > 600'

SELECT COUNT(*)

FROM prices

 WHERE WORLD > 600'

**domanda numero 2**

In [ ]:
df_clean[df_clean.index.year == 2000]['WORLD'].max()

135.66760743658767

SELECT MAX(WORLD)

FROM prices

 WHERE Date >= '2000-01-01' AND Date <= '2000-12-31"

**domanda numero 3**

In [ ]:
#opzione 1
df_clean.loc['2016-08-04':'2017-12-31', 'GOLD'].min()

#opzione 2
mask = (df_clean.index >= '2016-08-04') & (df_clean.index <= '2017-12-31')
df_clean.loc[mask, 'GOLD'].min()

#opzione 3
df_clean.loc[df_clean.index.to_series().between('2016-08-04', '2017-12-31'), 'GOLD'].min()

#opzione 4
df_clean.query("index >= '2016-08-04' and index <= '2017-12-31'")['GOLD'].min()

1055.47

SELECT MIN(GOLD)

FROM prices

WHERE Date BETWEEN '2016-08-04' AND '2017-12-31';

**domanda numero 4**

In [ ]:
# opzione 1, "resemple"
df_clean[['WORLD', 'WORLD_SMALL_CAP', 'GOLD']].resample('YE').mean()

# opzione 2, "group by"
df_clean[['WORLD', 'WORLD_SMALL_CAP', 'GOLD']].groupby(df_clean.index.year).mean()

,WORLD,WORLD_SMALL_CAP,GOLD
Data,,,
1998,NaN,NaN,248.580805
1999,NaN,NaN,261.697726
2000,135.667607,100.000000,302.464563
2001,125.332892,103.709779,303.138849
2002,100.640422,94.915420,328.742329
2003,83.117821,85.837998,321.425671
2004,93.901289,108.802875,329.476585
2005,106.312550,130.485660,358.983425
2006,122.685055,155.564050,481.104274


SELECT

    YEAR(Date) AS Anno,

    AVG(WORLD) AS avg_world,

    AVG(WORLD_SMALL_CAP) AS avg_small_cap,

    AVG(GOLD) AS avg_gold

FROM prices

GROUP BY Anno

ORDER BY Anno;

**domanda 5**

In [ ]:
df_clean['WORLD_SMALL_CAP'].groupby(df_clean.index.year).std().idxmax()

np.int32(2020)

-- funziona solo su MySQL e non su SQLite

SELECT

    YEAR(Date) AS Anno,

    STDDEV_SAMP(WORLD_SMALL_CAP) AS volatilita

FROM prices

GROUP BY YEAR(Date)

ORDER BY volatilita DESC

LIMIT 1;



**domanda 6**

In [ ]:
df_clean['WORLD'].corr(df_clean['GOLD'])

np.float64(0.9158935608453291)

SQLite non supporta la funzione CORR(), ne tantomeno MySQL. In Oracle si usa

SELECT CORR(WORLD, GOLD)

FROM prices;


Calcolarla in SQL puro su SQLite richiede la formula estesa di Pearson:$$r = \frac{\sum (x - \bar{x})(y - \bar{y})}{\sqrt{\sum (x - \bar{x})^2 \sum (y - \bar{y})^2}}$$

Risulta una query con CTE o subquery estremamente prolissa e poco usata nella pratica, motivo per cui per l'analisi esplorativa e statistica si delega quasi sempre a Python.

**domanda 7**

In [ ]:
# opzione 1
(df_clean['WORLD'].diff() > 0).sum()

# opzione 2:
(df_clean['WORLD'] > df_clean['WORLD'].shift(1)).sum()

WITH variazioni AS (

SELECT
  
  WORLD,
  
  LAG(WORLD, 1) OVER (ORDER BY Date) AS world_prec
  
FROM prices

)

SELECT COUNT(*)

FROM variazioni

WHERE WORLD > world_prec;